# 4. Linear algebra with symbols

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ktwyw/sympy-for-engineers/blob/main/notebooks/04_linear_algebra.ipynb)

NumPy matrices hold numbers. SymPy matrices can hold symbols, so determinants, inverses and
eigenvalues come out as formulas.

**Worked examples:** balancing a combustion reaction, currents in a circuit, natural frequencies
of a two-mass system, principal stresses, and the response of a linear system.

In [1]:
import sympy as sp

sp.init_printing(use_latex="mathjax")

## 4.1 Matrices

Matrix multiplication is `*` (not `@` as in NumPy, although `@` also works).

In [2]:
a, b, c, d = sp.symbols("a b c d")
M = sp.Matrix([[a, b], [c, d]])
M, M.det(), M.inv()

⎛                   ⎡    d         -b    ⎤⎞
⎜                   ⎢─────────  ─────────⎥⎟
⎜⎡a  b⎤             ⎢a⋅d - b⋅c  a⋅d - b⋅c⎥⎟
⎜⎢    ⎥, a⋅d - b⋅c, ⎢                    ⎥⎟
⎜⎣c  d⎦             ⎢   -c          a    ⎥⎟
⎜                   ⎢─────────  ─────────⎥⎟
⎝                   ⎣a⋅d - b⋅c  a⋅d - b⋅c⎦⎠

In [3]:
M.T, M * M, M.rank()

⎛        ⎡ 2                  ⎤   ⎞
⎜⎡a  c⎤  ⎢a  + b⋅c   a⋅b + b⋅d⎥   ⎟
⎜⎢    ⎥, ⎢                    ⎥, 2⎟
⎜⎣b  d⎦  ⎢                  2 ⎥   ⎟
⎝        ⎣a⋅c + c⋅d  b⋅c + d  ⎦   ⎠

## 4.2 Balancing a chemical reaction with a null space

**Problem.** Balance the combustion of propane:
$\;x_1\,\mathrm{C_3H_8} + x_2\,\mathrm{O_2} \rightarrow x_3\,\mathrm{CO_2} + x_4\,\mathrm{H_2O}$.

**By hand.** Each element gives one linear equation: atoms in = atoms out. Written as
$A\mathbf{x} = \mathbf{0}$, with products entered as negative, the coefficients are a vector
in the null space of $A$.

In [4]:
#               C3H8  O2  CO2  H2O
A = sp.Matrix([[3,    0,  -1,   0],     # carbon
               [8,    0,   0,  -2],     # hydrogen
               [0,    2,  -2,  -1]])    # oxygen

null_vector = A.nullspace()[0]
null_vector.T

[1/4  5/4  3/4  1]

In [5]:
# scale to the smallest whole numbers
coefficients = null_vector * sp.ilcm(*[term.q for term in null_vector])
assert list(coefficients) == [1, 5, 3, 4]
coefficients.T

[1  5  3  4]

$\mathrm{C_3H_8 + 5\,O_2 \rightarrow 3\,CO_2 + 4\,H_2O}$. The same three lines balance any
reaction; only the matrix changes. If the null space has more than one vector, the "reaction"
is really several independent reactions.

## 4.3 A symbolic linear system: a two-loop circuit

**Problem.** A source $V_s$ drives two meshes. Resistor $R_1$ is in the first mesh, $R_3$ in the
second, and $R_2$ is shared. Find the mesh currents.

**By hand.** Kirchhoff's voltage law around each mesh:

$$(R_1 + R_2)\,i_1 - R_2\,i_2 = V_s,\qquad -R_2\,i_1 + (R_2 + R_3)\,i_2 = 0$$

In [6]:
R1, R2, R3, Vs = sp.symbols("R_1 R_2 R_3 V_s", positive=True)

K = sp.Matrix([[R1 + R2, -R2], [-R2, R2 + R3]])
rhs = sp.Matrix([Vs, 0])
currents = sp.simplify(K.LUsolve(rhs))
currents

⎡       Vₛ⋅(R₂ + R₃)        ⎤
⎢───────────────────────────⎥
⎢    2                      ⎥
⎢- R₂  + (R₁ + R₂)⋅(R₂ + R₃)⎥
⎢                           ⎥
⎢           R₂⋅Vₛ           ⎥
⎢───────────────────────────⎥
⎢    2                      ⎥
⎣- R₂  + (R₁ + R₂)⋅(R₂ + R₃)⎦

**Verify.** The source sees $R_1$ in series with $R_2 \parallel R_3$, so $i_1$ must equal
$V_s / \left(R_1 + \dfrac{R_2 R_3}{R_2 + R_3}\right)$. And if $R_3 \to \infty$ (open circuit)
no current flows in the second mesh.

In [7]:
assert sp.simplify(currents[0] - Vs / (R1 + R2 * R3 / (R2 + R3))) == 0
assert sp.limit(currents[1], R3, sp.oo) == 0

## 4.4 Eigenvalues: natural frequencies of two coupled masses

**Problem.** Two equal masses $m$ sit between two walls, connected by three equal springs $k$
(wall–mass–mass–wall). Find the natural frequencies and mode shapes.

**By hand.** $\mathbf{M}\ddot{\mathbf{x}} + \mathbf{K}\mathbf{x} = 0$. Trying
$\mathbf{x} = \mathbf{u}\,e^{i\omega t}$ gives the eigenvalue problem
$\mathbf{M}^{-1}\mathbf{K}\,\mathbf{u} = \omega^2\mathbf{u}$.

In [8]:
m, k = sp.symbols("m k", positive=True)

M_mass = sp.diag(m, m)
K_stiff = sp.Matrix([[2 * k, -k], [-k, 2 * k]])

modes = (M_mass.inv() * K_stiff).eigenvects()     # (eigenvalue, multiplicity, [vectors])
for omega_sq, _, vectors in modes:
    display(sp.sqrt(omega_sq), vectors[0].T)

√k
──
√m

[1  1]

√3⋅√k
─────
 √m  

[-1  1]

The lower frequency $\sqrt{k/m}$ has both masses moving together $(1, 1)$, so the middle spring
is never stretched. The higher frequency $\sqrt{3k/m}$ has them moving in opposition $(-1, 1)$.

**Verify** through the characteristic equation $\det(\mathbf{K} - \omega^2\mathbf{M}) = 0$.

In [9]:
w = sp.symbols("omega", positive=True)
frequencies = sp.solve((K_stiff - w**2 * M_mass).det(), w)
assert set(frequencies) == {sp.sqrt(k / m), sp.sqrt(3 * k / m)}
frequencies

⎡√k  √3⋅√k⎤
⎢──, ─────⎥
⎣√m   √m  ⎦

## 4.5 Principal stresses

**Problem.** For a plane stress state $\sigma_x, \sigma_y, \tau_{xy}$, find the principal stresses.

**By hand.** They are the eigenvalues of the stress tensor, the formula behind Mohr's circle:
$\sigma_{1,2} = \dfrac{\sigma_x+\sigma_y}{2} \pm \sqrt{\left(\dfrac{\sigma_x-\sigma_y}{2}\right)^2 + \tau_{xy}^2}$.

In [10]:
sx, sy, txy = sp.symbols("sigma_x sigma_y tau_xy", real=True)
stress = sp.Matrix([[sx, txy], [txy, sy]])

principal = list(stress.eigenvals())
principal

⎡              _________________________________                ______________ ↪
⎢             ╱   2                 2         2                ╱   2           ↪
⎢σₓ   σ_y   ╲╱  σₓ  - 2⋅σₓ⋅σ_y + σ_y  + 4⋅τ_xy    σₓ   σ_y   ╲╱  σₓ  - 2⋅σₓ⋅σ_ ↪
⎢── + ─── - ────────────────────────────────────, ── + ─── + ───────────────── ↪
⎣2     2                     2                    2     2                      ↪

↪ ___________________⎤
↪        2         2 ⎥
↪ y + σ_y  + 4⋅τ_xy  ⎥
↪ ───────────────────⎥
↪ 2                  ⎦

In [11]:
radius = sp.sqrt(((sx - sy) / 2) ** 2 + txy**2)          # radius of Mohr's circle
centre = (sx + sy) / 2
assert all(sp.simplify((p - centre) ** 2 - radius**2) == 0 for p in principal)   # centre ± radius

# numbers, in MPa
state = {sx: 80, sy: -40, txy: 30}
sorted((float(p.subs(state)) for p in principal), reverse=True)

[87.08203932499369, -47.082039324993694]

A three-dimensional numerical state works the same way. For a symmetric matrix the
eigenvectors are the principal directions.

In [12]:
stress_3d = sp.Matrix([[50, 30, 0], [30, -20, 0], [0, 0, 10]])
{sp.N(val, 5): mult for val, mult in stress_3d.eigenvals().items()}

{-31.098: 1, 10.0: 1, 61.098: 1}

## 4.6 Rotation matrices

Symbolic matrices make it easy to check identities that are tedious by hand.

In [13]:
alpha, beta = sp.symbols("alpha beta", real=True)

def rot(angle):
    return sp.Matrix([[sp.cos(angle), -sp.sin(angle)], [sp.sin(angle), sp.cos(angle)]])

assert sp.simplify(rot(alpha) * rot(beta) - rot(alpha + beta)) == sp.zeros(2)   # rotations add
assert sp.simplify(rot(alpha).det()) == 1                                       # no stretching
assert sp.simplify(rot(alpha).inv() - rot(alpha).T) == sp.zeros(2)              # inverse = transpose
sp.simplify(rot(alpha) * rot(beta))

⎡cos(α + β)  -sin(α + β)⎤
⎢                       ⎥
⎣sin(α + β)  cos(α + β) ⎦

## 4.7 The matrix exponential: solving $\dot{\mathbf{x}} = \mathbf{A}\mathbf{x}$

A linear system of differential equations has the solution $\mathbf{x}(t) = e^{\mathbf{A}t}\mathbf{x}_0$.
Here $\mathbf{A}$ describes a mass–spring–damper with $\ddot x + 3\dot x + 2x = 0$,
written with the state $(x, \dot x)$.

In [14]:
t = sp.symbols("t", nonnegative=True)
A_sys = sp.Matrix([[0, 1], [-2, -3]])

A_sys.eigenvals()        # both negative: the system is stable and does not oscillate

{-2: 1, -1: 1}

In [15]:
x0 = sp.Matrix([1, 0])                       # released from rest at x = 1
x_t = sp.simplify(sp.exp(A_sys * t) * x0)
x_t

⎡⎛   t    ⎞  -2⋅t⎤
⎢⎝2⋅ℯ  - 1⎠⋅ℯ    ⎥
⎢                ⎥
⎢  ⎛     t⎞  -2⋅t⎥
⎣2⋅⎝1 - ℯ ⎠⋅ℯ    ⎦

In [16]:
# verify: it satisfies the differential equation and the initial condition
assert sp.simplify(sp.diff(x_t, t) - A_sys * x_t) == sp.zeros(2, 1)
assert x_t.subs(t, 0) == x0

## Exercises

1. Balance $\mathrm{C_2H_5OH + O_2 \rightarrow CO_2 + H_2O}$ and
   $\mathrm{KMnO_4 + HCl \rightarrow KCl + MnCl_2 + H_2O + Cl_2}$ with the null-space method.
2. In 4.4, make the middle spring $k_c$ different from the outer ones. How do the two
   frequencies depend on $k_c$? What happens as $k_c \to 0$?
3. Show that the maximum shear stress in plane stress is half the difference of the principal
   stresses.
4. Write the 3D rotation matrices about $x$ and $z$, and show that they do not commute.
5. Repeat 4.7 for $\ddot x + 2\dot x + x = 0$, whose matrix is `[[0, 1], [-1, -2]]`. The
   eigenvalue is repeated (critical damping). How does the form of the solution change?